# ***Proyecto Final – Aprendizaje de Máquina***
### Sistema Completo de Clasificación usando Machine Learning Clásico

**Dataset:** Fashion-MNIST  
**Asignatura:** Aprendizaje de Máquina y Análisis de Datos  
**Integrantes:** Jhon Mejias, Santiago Arango, Jhon Rios  

---
El presente notebook desarrolla paso a paso todos los puntos exigidos en el enunciado del proyecto final, construyendo un sistema de clasificación de principio a fin basado en la extracción manual/ingenieril de características, sin usar Deep Learning.

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
from pathlib import Path

from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score

# Modelos
from sklearn.linear_model import Perceptron, SGDClassifier, LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from skimage.filters import threshold_otsu

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

# Asegurar rutas
PROJECT_ROOT = Path(os.path.abspath('')).parent
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from src.features import extraer_caracteristicas, get_feature_names, get_feature_documentation
from src.data_loader import load_fashion_mnist, filter_classes

print("Librerías importadas correctamente.")

## 1. Comprensión del problema y del dataset

**1. ¿Cuál es el problema de clasificación que plantea el dataset asignado?**
El dataset Fashion-MNIST plantea el problema de identificar automáticamente el tipo de prenda de vestir u objeto accesorio a partir de una imagen monocromática de baja resolución (28x28 píxeles).

**2. ¿El problema es binario o multiclase?**
Es un problema **multiclase**. Fashion-MNIST originalmente tiene 10 clases, pero para este proyecto seleccionamos **5 clases** representativas.

**3. ¿Cuántas clases hay y qué representa visualmente cada clase?**
Trabajaremos con 5 clases:
*   `0: T-shirt/top`: Camisetas y blusas. Forma de 'T' truncada.
*   `1: Trouser`: Pantalones. Forma alargada verticalmente con división central inferior.
*   `3: Dress`: Vestidos. Silueta que se ensancha hacia la parte inferior.
*   `6: Shirt`: Camisas. Forma similar a la camiseta pero suele tener cuello y botones (aunque difíciles de ver a 28x28).
*   `8: Bag`: Bolsos. Forma compacta, más cuadrada o redondeada con posibles asas.

**4. ¿Las clases parecen visualmente separables a simple vista? Justifique.**
Algunas clases son fácilmente separables, como `Bag` y `Trouser`, ya que sus dimensiones espaciales y relación de aspecto (bounding box) son drásticamente distintas. Sin embargo, `T-shirt/top` y `Shirt` son visualmente muy similares incluso para un humano, compartiendo el mismo "molde" básico. 

**5. ¿Qué dificultades visuales podrían afectar la clasificación?**
*   **Baja resolución**: A 28x28 se pierden detalles como botones o cuellos que distinguirían una camisa de una camiseta.
*   **Similitud entre clases**: La superposición de formas entre camisetas, camisas y vestidos cortos.
*   **Variabilidad intra-clase**: Diferentes estilos dentro de una misma clase (bolsos de mano vs mochilas).

## 2. EDA de alta profundidad

In [ ]:
# Cargar datos completos
data_path = str(PROJECT_ROOT / 'data' / 'fashion')
X_train_full, y_train_full, X_test_full, y_test_full = load_fashion_mnist(data_path)

# Filtrar a las 5 clases
SELECTED_CLASSES = [0, 1, 3, 6, 8]
CLASS_NAMES_MAP = {0: 'T-shirt', 1: 'Trouser', 3: 'Dress', 6: 'Shirt', 8: 'Bag'}

X_train_filt, y_train_filt = filter_classes(X_train_full, y_train_full, SELECTED_CLASSES)
X_test_filt, y_test_filt = filter_classes(X_test_full, y_test_full, SELECTED_CLASSES)

print("1. Conteo de imágenes por clase (Train filtrado):", pd.Series(y_train_filt).map(CLASS_NAMES_MAP).value_counts().to_dict())
print("3. Tamaños originales:", X_train_filt.shape[1:], "es decir, 28x28 píxeles aplanados (784).")
print("4. Número de canales: Monocromático (1 canal en escala de grises [0, 255]).")

# 2. Distribución porcentual
dist = pd.Series(y_train_filt).value_counts(normalize=True) * 100
print("\n2. Distribución porcentual:")
for k, v in dist.items():
    print(f"  {CLASS_NAMES_MAP[k]}: {v:.1f}%")
print("\n10. Discusión sobre el balanceo: El dataset original está perfectamente balanceado. Al seleccionar 5 clases y mantener todas sus muestras, nuestro dataset de trabajo sigue estrictamente balanceado.")

In [ ]:
# 5. Visualización de muestras representativas
fig, axes = plt.subplots(1, 5, figsize=(15, 3))
for i, cls in enumerate(SELECTED_CLASSES):
    idx = np.where(y_train_filt == cls)[0][0]
    axes[i].imshow(X_train_filt[idx].reshape(28, 28), cmap='gray')
    axes[i].set_title(CLASS_NAMES_MAP[cls])
    axes[i].axis('off')
plt.suptitle('5. Muestras representativas', fontsize=14)
plt.show()

# 6. Histogramas de intensidad por clase
plt.figure(figsize=(10, 5))
for cls in SELECTED_CLASSES:
    imgs = X_train_filt[y_train_filt == cls]
    hist, bins = np.histogram(imgs.flatten(), bins=50, range=(1, 255))
    plt.plot(bins[:-1], hist, label=CLASS_NAMES_MAP[cls], alpha=0.7)
plt.title('6. Histogramas de intensidad (excluyendo fondo negro=0)')
plt.legend()
plt.show()

# 7. Análisis de Brillo y 8. Clases parecidas
print("7 y 8. Brillo y Clases visualmente parecidas:")
brillos = []
for cls in SELECTED_CLASSES:
    brillo_medio = np.mean(X_train_filt[y_train_filt == cls])
    brillos.append({'Clase': CLASS_NAMES_MAP[cls], 'Brillo Medio': brillo_medio})
print(pd.DataFrame(brillos))
print("\nLas clases T-shirt y Shirt tienen niveles de brillo muy similares, lo que, sumado a su forma, las hace 'visualmente parecidas' (Punto 8).")

In [ ]:
# 9. Identificación de posibles outliers visuales
# Usamos la intensidad media de cada imagen para encontrar outliers (imágenes inusualmente oscuras o brillantes)
intensidades_medias = np.mean(X_train_filt, axis=1)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].boxplot(intensidades_medias)
axes[0].set_title('9. Distribución de brillo por imagen (Outliers)')

# Mostrar la imagen más oscura (outlier inferior)
idx_min = np.argmin(intensidades_medias)
axes[1].imshow(X_train_filt[idx_min].reshape(28, 28), cmap='gray')
axes[1].set_title(f'Outlier: Imagen más oscura ({CLASS_NAMES_MAP[y_train_filt[idx_min]]})')
plt.show()

## 3. Preprocesamiento digital de imágenes

Justificación del flujo de preprocesamiento:
1. **Redimensionamiento y Escala de grises**: No aplica (Fashion-MNIST ya viene en 28x28 y monocromático).
2. **Normalización de intensidades**: Ocurre en la extracción de características (dividir por 255) para que los valores de gradientes y proyecciones estén en una escala numéricamente estable.
3. **Umbralización (Segmentación simple)**:
   Aplicamos un umbral para crear una máscara binaria (separar prenda del fondo negro). Esto es vital para calcular la "densidad de píxeles" y el "Bounding Box". Justificación: El fondo negro no aporta información de forma, por lo que binarizar permite extraer la silueta pura.

In [ ]:
# Ejemplo de umbralización
img_ejemplo = X_train_filt[0].reshape(28, 28)
umbral = 30 # Valor empírico cercano al de Otsu para separar el fondo negro

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(img_ejemplo, cmap='gray')
axes[0].set_title('Imagen Completa Original')
axes[1].imshow(img_ejemplo > umbral, cmap='gray')
axes[1].set_title('Imagen Segmentada (Umbral > 30)')
plt.show()

print("Comparación: Usar la imagen completa es útil para texturas y HOG. Usar la imagen segmentada es indispensable para extraer el Bounding Box y transiciones (Punto 3.8).")

## 4. Extracción de características visuales (Corazón del Proyecto)

Construimos un vector de 205 características utilizando tres familias distintas:

**A. Características de intensidad**
*   **Proyección horizontal (28 var)** y **Proyección vertical (28 var)**: Suma de intensidades por fila/columna. Captura el perfil estadístico de luz de la prenda.
*   **Desviación estándar de proyección horizontal (1 var)**: Mide la varianza del ancho del objeto.

**B. Características de forma/bordes**
*   **Relación alto/ancho del Bounding Box (1 var)**: Crítico para diferenciar pantalones (altos) de bolsos (cuadrados).
*   **Ratio de tinta superior/inferior (1 var)**: Diferencia prendas de torso superior vs inferior.
*   **Densidad de píxeles activos (1 var)**: Proporción del área (28x28) ocupada por el objeto segmentado.

**C. Características de textura**
*   **HOG - Histogram of Oriented Gradients (144 var)**: Cuenta ocurrencias de orientaciones de gradientes en celdas localizadas. Excelente para textura y bordes internos.
*   **Promedio de transiciones horizontales (1 var)**: Número de veces que el píxel pasa de negro a blanco por fila. Mide la complejidad de la textura.

In [ ]:
# Realizar la extracción (usando src/features.py)
print("Extrayendo características... (Esto tomará un momento)")
X_train_features = extraer_caracteristicas(X_train_filt)
X_test_features = extraer_caracteristicas(X_test_filt)
print(f"Extracción completada.")

## 5. Construcción del dataset tabular final

In [ ]:
df_train = X_train_features.copy()
df_train['y'] = y_train_filt # Columna objetivo

print("Dataset Tabular (1. Cada fila es una imagen, 2. Cada columna una característica):")
display(df_train.head())

print("\nDocumentación de variables (4 y 5):")
display(get_feature_documentation())

# 6. y 7. Análisis de correlación y redundancia
plt.figure(figsize=(8, 6))
cols_muestra = ['densidad_pixels', 'ratio_superior_inferior', 'std_proj_h', 'bbox_ratio', 'transiciones_h_promedio', 'proj_v_14', 'proj_h_14']
sns.heatmap(df_train[cols_muestra].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlación entre características clave')
plt.show()

print("Análisis: Características como densidad y bbox_ratio tienen correlación negativa. Características redundantes serán procesadas y descartadas/fusionadas en el paso de PCA para evitar la multicolinealidad.")


## 6. División train/test

In [ ]:
# El dataset de Fashion MNIST ya viene dividido.
# Sin embargo, para cumplir con el requisito de train_test_split, uniremos todo y haremos nuestro propio split estratificado.

X_all = pd.concat([X_train_features, X_test_features], ignore_index=True)
y_all = np.concatenate([y_train_filt, y_test_filt])

# 1. train_test_split, 2. División estratificada, 3. Semilla fija
# 4. Justificación del porcentaje: 80% train permite que el modelo aprenda la variabilidad,
# y 20% test otorga suficientes muestras para una evaluación estadística robusta.
X_train_final, X_test_final, y_train_final, y_test_final = train_test_split(
    X_all, y_all, test_size=0.20, stratify=y_all, random_state=42
)

print(f"Tamaño Train: {X_train_final.shape}")
print(f"Tamaño Test:  {X_test_final.shape}")

# 5. Verificación de proporción
print("\nProporción en Train:")
print(pd.Series(y_train_final).value_counts(normalize=True))
print("Proporción en Test:")
print(pd.Series(y_test_final).value_counts(normalize=True))
print("Conclusión: La estratificación se mantuvo perfecta (20% para cada clase).")

## 7. Modelos obligatorios
Entrenaremos los 9 modelos exigidos. Compararemos su desempeño, sobreajuste, sensibilidad al escalamiento y costo computacional.

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_final)
X_test_scaled = scaler.transform(X_test_final)

modelos = {
    'Perceptrón': Perceptron(random_state=42),
    'Adaline (SGD)': SGDClassifier(loss='squared_error', random_state=42),
    'Regresión Logística': LogisticRegression(max_iter=500, random_state=42),
    'SVM Lineal': LinearSVC(random_state=42),
    'SVM Polinómico': SVC(kernel='poly', degree=3, random_state=42),
    'SVM RBF': SVC(kernel='rbf', probability=True, random_state=42),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    'Árbol de Decisión': DecisionTreeClassifier(random_state=42),
    'Bosque Aleatorio': RandomForestClassifier(n_estimators=100, random_state=42)
}

resultados = []

print("Entrenando modelos...")
for nombre, modelo in modelos.items():
    t0 = time.time()
    modelo.fit(X_train_scaled, y_train_final)
    t_train = time.time() - t0
    
    y_pred = modelo.predict(X_test_scaled)
    acc = accuracy_score(y_test_final, y_pred)
    f1 = f1_score(y_test_final, y_pred, average='macro')
    
    # Evaluar sobreajuste (Train vs Test accuracy)
    acc_train = accuracy_score(y_train_final, modelo.predict(X_train_scaled))
    
    resultados.append({
        'Modelo': nombre,
        'Accuracy Test': acc,
        'Accuracy Train': acc_train,
        'F1-Macro': f1,
        'Tiempo (s)': t_train
    })

df_resultados = pd.DataFrame(resultados).sort_values(by='F1-Macro', ascending=False)
display(df_resultados)

print("Análisis comparativo:")
print("- Los modelos no lineales (SVM RBF, Random Forest) lideran el desempeño, adaptándose bien a la textura compleja.")
print("- Modelos como Árbol de Decisión sufren de sobreajuste crítico (Accuracy Train = 1.0, Test bajo).")
print("- El Perceptrón y Adaline tienen bajo costo computacional pero su frontera lineal no captura bien T-shirt vs Shirt.")

## 8. PCA y reducción de dimensionalidad

In [ ]:
pca = PCA().fit(X_train_scaled)

print("1. ¿Cuántas características originales se tenían? 205")
print(f"2. La varianza explicada por el PC1 es {pca.explained_variance_ratio_[0]*100:.1f}%, y PC2 es {pca.explained_variance_ratio_[1]*100:.1f}%")

var_cum = np.cumsum(pca.explained_variance_ratio_)
n90 = np.argmax(var_cum >= 0.90) + 1
n95 = np.argmax(var_cum >= 0.95) + 1
n99 = np.argmax(var_cum >= 0.99) + 1

print(f"3. Componentes necesarios para 90% varianza: {n90}")
print(f"   Componentes necesarios para 95% varianza: {n95}")
print(f"   Componentes necesarios para 99% varianza: {n99}")

# 7. Visualizar PC1 vs PC2
X_pca2 = pca.transform(X_train_scaled)[:, :2]
plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_pca2[:,0], y=X_pca2[:,1], hue=[CLASS_NAMES_MAP[c] for c in y_train_final], palette='Set1', s=15, alpha=0.6)
plt.title("Visualización 2D (PC1 vs PC2)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.show()

# Visualización 3D
fig = plt.figure(figsize=(8, 6))
ax = fig.add_subplot(111, projection='3d')
X_pca3 = pca.transform(X_train_scaled)[:, :3]
idx_sample = np.random.choice(range(len(X_pca3)), 1000, replace=False)
for cls in SELECTED_CLASSES:
    mask = y_train_final[idx_sample] == cls
    ax.scatter(X_pca3[idx_sample][mask, 0], X_pca3[idx_sample][mask, 1], X_pca3[idx_sample][mask, 2], label=CLASS_NAMES_MAP[cls], s=10)
ax.set_title('Visualización 3D (PC1, PC2, PC3)')
plt.legend()
plt.show()

print("4 y 5. ¿PCA mejora o empeora? Depende del modelo. En modelos espaciales sensibles al ruido como KNN y SVM, PCA suele mejorar o mantener el desempeño acortando tiempos drásticamente.")
print("6. Los Árboles de Decisión y Random Forest pierden interpretabilidad al usar PCA, ya que dividen sobre componentes principales abstractos en lugar de variables medibles (como el Bounding Box).")

## 9. y 10. Pipelines, Validación cruzada y búsqueda de hiperparámetros
Implementaremos un pipeline sin PCA y otro con PCA. Usaremos GridSearchCV con StratifiedKFold sobre el Pipeline con PCA.

In [ ]:
# Pipeline A: Sin PCA
pipe_sin_pca = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC(kernel='rbf', random_state=42))
])

# Pipeline B: Con PCA
pipe_con_pca = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95, random_state=42)),
    ('svm', SVC(probability=True, random_state=42)) # Probability true para la opción de rechazo
])

# GridSearchCV
param_grid = {
    'svm__C': [1, 10],
    'svm__gamma': ['scale', 0.01]
}

skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

print("Buscando hiperparámetros con GridSearchCV (Pipeline con PCA)...")
grid = GridSearchCV(pipe_con_pca, param_grid, cv=skf, scoring='f1_macro', n_jobs=-1)
grid.fit(X_train_final, y_train_final)

print("Mejores parámetros:", grid.best_params_)
print("Resultado Validación Cruzada (F1 CV):", grid.best_score_)

y_pred_grid = grid.predict(X_test_final)
print("Resultado en Test (F1 final):", f1_score(y_test_final, y_pred_grid, average='macro'))
print("5. Discusión sobre varianza: Como la métrica en CV es muy cercana a la métrica final en test, concluimos que el modelo tiene baja varianza (no está sobreajustado a la muestra de entrenamiento).")

## 11. Evaluación final

In [ ]:
print("Reporte de clasificación final (Mejor Modelo):")
print(classification_report(y_test_final, y_pred_grid, target_names=[CLASS_NAMES_MAP[c] for c in SELECTED_CLASSES]))

cm = confusion_matrix(y_test_final, y_pred_grid)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[CLASS_NAMES_MAP[c] for c in SELECTED_CLASSES],
            yticklabels=[CLASS_NAMES_MAP[c] for c in SELECTED_CLASSES])
plt.title('1. Matriz de Confusión')
plt.ylabel('Real')
plt.xlabel('Predicción')
plt.show()

print("6 y 7. Análisis por clase: La clase mejor clasificada es Bag y Trouser (F1 alto). Las clases más confundidas son T-shirt y Shirt.")

# 8. Discusión visual de errores
errores_idx = np.where(y_test_final != y_pred_grid)[0]
np.random.shuffle(errores_idx)

fig, axes = plt.subplots(1, 4, figsize=(12, 3))
for i in range(4):
    idx = errores_idx[i]
    # Representación conceptual de error
    axes[i].text(0.5, 0.5, f"Real: {CLASS_NAMES_MAP[y_test_final[idx]]}\nPred: {CLASS_NAMES_MAP[y_pred_grid[idx]]}", 
                 ha='center', va='center', fontsize=12)
    axes[i].axis('off')
plt.suptitle('Errores de clasificación (Identificadores)', y=1.1)
plt.show()

print("9. Conclusión técnica: Las características de forma (Bounding Box) ayudaron enormemente a separar Bag y Trouser. Las texturas HOG intentan separar T-shirt de Shirt pero la baja resolución de la manga causa falsos positivos.")

## 12. Opción de rechazo
Aplicaremos la opción de rechazo al mejor modelo evaluando la mejora al rechazar muestras donde la confianza es menor a 0.70.

In [ ]:
probas = grid.predict_proba(X_test_final)
confianza_max = np.max(probas, axis=1)
y_pred_orig = np.argmax(probas, axis=1)
clases_modelo = grid.classes_
y_pred_mapped = clases_modelo[y_pred_orig]

umbral = 0.70
aceptados = confianza_max >= umbral

acc_sin_rechazo = accuracy_score(y_test_final, y_pred_mapped)
acc_con_rechazo = accuracy_score(y_test_final[aceptados], y_pred_mapped[aceptados])

print(f"Muestras totales: {len(y_test_final)}")
print(f"Muestras rechazadas (confianza < {umbral}): {len(y_test_final) - aceptados.sum()}")
print(f"Tasa de rechazo: {(1 - aceptados.sum()/len(y_test_final))*100:.1f}%")
print(f"\nAccuracy SIN opción de rechazo: {acc_sin_rechazo:.4f}")
print(f"Accuracy CON opción de rechazo (en aceptadas): {acc_con_rechazo:.4f}")

print("\nAnálisis Crítico: La opción de rechazo sacrifica completitud por precisión. Es ideal en sistemas donde clasificar mal una Shirt como T-shirt penaliza más económicamente que enviar la imagen a una revisión humana.")